# 🤖 AI Agent Excel Assistant — Experimentation Notebook

This notebook demonstrates the full workflow: loading data, defining tools,
building the agent, and testing different scenarios.

## Setup
Make sure you have:
1. Installed dependencies: `pip install -r requirements.txt`
2. Configured `.env` with at least one LLM API key

In [1]:
import sys
sys.path.insert(0, '..')

# Verify imports
from app.config import settings
print(f"✅ Config loaded — Provider: {settings.llm_provider.value}, Model: {settings.active_model}")

✅ Config loaded — Provider: github_models, Model: gpt-4o


## 1. Loading and Exploring Data

The DataManager loads both Excel files into pandas DataFrames on initialization.

In [2]:
from app.data.manager import DataManager

dm = DataManager()

# List all available datasets
for ds in dm.list_datasets():
    print(f"📊 {ds['display_name']}: {ds['rows']} rows, {len(ds['columns'])} columns")
    print(f"   Columns: {ds['columns']}")
    print()

📊 Real Estate Listings: 1005 rows, 11 columns
   Columns: ['Listing ID', 'Property Type', 'City', 'State', 'Bedrooms', 'Bathrooms', 'Square Footage', 'Year Built', 'List Price', 'Sale Price', 'Listing Status']

📊 Marketing Campaigns: 998 rows, 11 columns
   Columns: ['Campaign ID', 'Campaign Name', 'Channel', 'Start Date', 'End Date', 'Budget Allocated', 'Amount Spent', 'Impressions', 'Clicks', 'Conversions', 'Revenue Generated']



In [3]:
# Inspect the Real Estate Listings schema
import json

schema = dm.get_schema("real_estate_listings")
print(f"Dataset: {schema['display_name']}")
print(f"Total rows: {schema['total_rows']}")
print(f"ID column: {schema['id_column']}")
print("\nColumns:")
for col in schema['columns']:
    info = f"  {col['name']} ({col['type']})"
    if 'unique_values' in col:
        info += f" — values: {col['unique_values']}"
    elif 'min' in col:
        avg = col.get('mean') or col.get('avg')
        info += f" — range: {col['min']:.0f} to {col['max']:.0f}"
        if avg is not None:
            info += f", avg: {avg:.0f}"
    print(info)

Dataset: Real Estate Listings
Total rows: 1005
ID column: Listing ID

Columns:
  Listing ID (text)
  Property Type (text) — values: ['Apartment', 'Condo', 'House', 'Townhouse']
  City (text)
  State (text) — values: ['Arizona', 'California', 'Colorado', 'Florida', 'Georgia', 'Illinois', 'Massachusetts', 'New York', 'Texas', 'Washington']
  Bedrooms (integer) — range: 1 to 5
  Bathrooms (decimal) — range: 1 to 4
  Square Footage (integer) — range: 240 to 2970
  Year Built (integer) — range: 1960 to 2025
  List Price (integer) — range: 33000 to 99999999
  Sale Price (decimal) — range: 34000 to 1853000
  Listing Status (text) — values: ['Active', 'Pending', 'Sold']


In [4]:
# Quick query example — how many listings per state (top 10)
result = dm.query(
    dataset="real_estate_listings",
    aggregation={"function": "count", "group_by": "State"}
)
# Sort by count descending
sorted_data = sorted(result['data'], key=lambda x: x.get('count', 0), reverse=True)[:10]
print("Top 10 States by Listing Count:")
for row in sorted_data:
    print(f"  {row['State']}: {row['count']} listings")

Top 10 States by Listing Count:
  Illinois: 116 listings
  Georgia: 113 listings
  Massachusetts: 108 listings
  Washington: 106 listings
  Florida: 98 listings
  Arizona: 96 listings
  Texas: 95 listings
  Colorado: 94 listings
  New York: 91 listings
  California: 88 listings


## 2. Defining and Testing Individual Tools

Each tool is self-contained with a name, description, parameter schema, and execute method.

In [5]:
from app.tools.base import ToolRegistry
from app.tools.query import QueryTool
from app.tools.insert import InsertTool
from app.tools.update import UpdateTool
from app.tools.delete import DeleteTool
from app.tools.schema_inspect import SchemaInspectTool
from app.tools.undo import UndoTool
from app.tools.list_changes import ListChangesTool
from app.data.validator import Validator

validator = Validator()
registry = ToolRegistry()

# Register all tools
registry.register(QueryTool(dm))
registry.register(InsertTool(dm, validator))
registry.register(UpdateTool(dm, validator))
registry.register(DeleteTool(dm))
registry.register(SchemaInspectTool(dm))
registry.register(UndoTool(dm))
registry.register(ListChangesTool(dm))

print(f"✅ Registered {len(registry.list_names())} tools: {registry.list_names()}")
print("\nTool Schemas (what the LLM sees):")
for schema in registry.get_schemas():
    print(f"  🔧 {schema['name']}: {schema['description'][:80]}...")

✅ Registered 7 tools: ['query_data', 'insert_data', 'update_data', 'delete_data', 'inspect_schema', 'undo_change', 'list_changes']

Tool Schemas (what the LLM sees):
  🔧 query_data: Query and search data from a dataset. Supports filtering by column values, sorti...
  🔧 insert_data: Insert new rows into a dataset. Validates data types, required fields, and value...
  🔧 update_data: Update existing rows in a dataset. Identifies rows using filters, validates new ...
  🔧 delete_data: Delete rows from a dataset. Identifies rows using filters, shows a preview of ro...
  🔧 inspect_schema: Inspect the structure and metadata of available datasets. Returns column names, ...
  🔧 undo_change: Undo a previous data mutation (insert, update, or delete). Can undo the most rec...
  🔧 list_changes: List recent data mutations (inserts, updates, deletes) from the change log. Show...


In [6]:
# Test QueryTool — simple filter
query_tool = registry.get("query_data")

result = query_tool.execute(
    dataset="real_estate_listings",
    filters=[{"column": "State", "operator": "eq", "value": "California"}],
    aggregation={"function": "count"}
)
print(f"Listings in California: {result.data['value']}")
print(f"Success: {result.success}")

Listings in California: 88
Success: True


In [7]:
# Test QueryTool — complex aggregation
result = query_tool.execute(
    dataset="real_estate_listings",
    filters=[
        {"column": "Property Type", "operator": "eq", "value": "House"},
        {"column": "Listing Status", "operator": "eq", "value": "Sold"}
    ],
    aggregation={"function": "avg", "column": "Sale Price", "group_by": "State"}
)
print("Average sale price of sold houses by state:")
sorted_data = sorted(result.data['data'], key=lambda x: x.get('avg_Sale Price', 0), reverse=True)[:5]
for row in sorted_data:
    print(f"  {row['State']}: ${row['avg_Sale Price']:,.0f}")

Average sale price of sold houses by state:
  Colorado: $1,006,421
  California: $954,706
  Washington: $857,400
  New York: $827,889
  Massachusetts: $809,500


In [8]:
# Test QueryTool — marketing campaigns, top by revenue
result = query_tool.execute(
    dataset="marketing_campaigns",
    sort_by="Revenue Generated",
    sort_order="desc",
    limit=5,
    columns=["Campaign ID", "Campaign Name", "Channel", "Revenue Generated"]
)
print("Top 5 Campaigns by Revenue:")
for row in result.data['data']:
    print(f"  {row['Campaign ID']}: {row['Campaign Name']} ({row['Channel']}) — ${row['Revenue Generated']:,.2f}")

Top 5 Campaigns by Revenue:
  CMP-8927: Summer Promo - Email 2025 Q1 (Email) — $503,194.13
  CMP-8682: Seasonal Collection - Email 2025 Q4 (Email) — $428,710.60
  CMP-8100: New Product Launch - Email 2025 Q3 (Email) — $424,886.78
  CMP-8696: Seasonal Collection - Email 2025 Q2 (Email) — $406,089.71
  CMP-8613: Brand Awareness - Email 2024 Q3 (Email) — $386,281.55


In [9]:
# Test Validator — valid data
valid_result = validator.validate_insert("real_estate_listings", [{
    "Listing ID": "LST-9999",
    "Property Type": "House",
    "City": "San Francisco",
    "State": "California",
    "Bedrooms": 3,
    "Bathrooms": 2.0,
    "Square Footage": 1500,
    "Year Built": 2020,
    "List Price": 850000,
    "Sale Price": 0,
    "Listing Status": "Active"
}])
print(f"Valid data: is_valid={valid_result.is_valid}, errors={valid_result.errors}")

# Test Validator — invalid data
invalid_result = validator.validate_insert("real_estate_listings", [{
    "Listing ID": "LST-9998",
    "Property Type": "Mansion",  # Invalid enum
    "Bedrooms": -1,  # Out of range
    "Year Built": 1600,  # Too old
}])
print(f"\nInvalid data: is_valid={invalid_result.is_valid}")
for err in invalid_result.errors:
    print(f"  ❌ {err}")

Valid data: is_valid=True, errors=[]

Invalid data: is_valid=False
  ❌ Row 1: Missing required field 'City'
  ❌ Row 1: Missing required field 'State'


In [10]:
# Test UpdateTool — preview without applying
update_tool = registry.get("update_data")

result = update_tool.execute(
    dataset="real_estate_listings",
    filters=[{"column": "Listing ID", "operator": "eq", "value": "LST-5001"}],
    updates={"List Price": 482000}
)
print(result.message)
print(f"\nRequires confirmation: {result.requires_confirmation}")

📊 CONFIRMATION: STAGED EXCEL UPDATE (1 Row)
──────────────────────────────────────────────────
📁 File: Real Estate Listings.xlsx

 Row  | ID        | Field        | Before       → After
──────────────────────────────────────────────────
 1    | LST-5001  | List Price   | 99,999,999   → 482,000

Apply this change? (yes/no)

Requires confirmation: True


## 3. Building the Agent

The Agent combines the LLM provider, tool registry, and data manager into a
ReAct reasoning loop.

In [11]:
from app.llm.base import get_provider
from app.agent.core import Agent
from app.agent.session import SessionManager
from app.logging.logger import InteractionLogger

# Initialize LLM provider
llm = get_provider(
    provider_name=settings.llm_provider.value,
    api_key=settings.active_api_key,
    model=settings.active_model,
)
print(f"🤖 LLM Provider: {llm}")

# Build agent
agent = Agent(
    llm=llm,
    tool_registry=registry,
    data_manager=dm,
    logger=InteractionLogger(),
)

session_mgr = SessionManager()
print("✅ Agent ready!")

🤖 LLM Provider: GitHubModelsProvider(model='gpt-4o')
✅ Agent ready!


## 4. Testing the Agent — End-to-End Scenarios

In [12]:
import asyncio

async def ask(question: str, session=None):
    """Helper to send a question to the agent and display the response."""
    if session is None:
        session = session_mgr.create_session()

    result = await agent.process_message(session, question)

    # ── Header ────────────────────────────────────────────────────────────────
    print(f"\n{'─' * 62}")
    print(f"❓  {question}")
    print(f"{'─' * 62}")

    # ── Answer ────────────────────────────────────────────────────────────────
    print(f"💬  {result.response}")

    # ── Tools + reasoning (merged, no redundancy) ─────────────────────────────
    if result.tool_calls:
        tools = [t['tool'] for t in result.tool_calls]
        steps = result.reasoning_steps or []
        step_count = len(steps)

        print(f"\n🔧  Tools: {', '.join(tools)}  ({step_count} step{'s' if step_count != 1 else ''})")
        for step in steps:
            if step.get('action'):
                # Strip JSON args — show only the tool name
                raw = step['action']
                tool_name = raw.split('(')[0].strip() if '(' in raw else raw.strip()
                print(f"     └─ Step {step['step']}: {tool_name}")

    # ── Confirmation flag ─────────────────────────────────────────────────────
    if result.requires_confirmation:
        print(f"\n⚠️   Awaiting confirmation — reply 'yes' to proceed")

    print()
    return session, result

In [13]:
# Scenario 1: Simple count query
session, _ = await ask("How many listings are in California?")

08:38:07 | DEBUG   | httpcore.connection | connect_tcp.started host='models.inference.ai.azure.com' port=443 local_address=None timeout=60.0 socket_options=None
08:38:07 | DEBUG   | httpcore.connection | connect_tcp.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001CB17231940>
08:38:07 | DEBUG   | httpcore.connection | start_tls.started ssl_context=<ssl.SSLContext object at 0x000001CB148DF610> server_hostname='models.inference.ai.azure.com' timeout=60.0
08:38:07 | DEBUG   | httpcore.connection | start_tls.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001CB170BF890>
08:38:07 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:38:07 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:38:07 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:38:07 | DEBUG   | httpcore.http11 | send_request_body.complete
08:38:07 | DEBUG   | httpcore.http11 | receive_


──────────────────────────────────────────────────────────────
❓  How many listings are in California?
──────────────────────────────────────────────────────────────
💬  There are **88 listings** in California.

🔧  Tools: query_data  (2 steps)
     └─ Step 1: query_data



In [14]:
# Scenario 2: Complex aggregation
_, _ = await ask("What is the average sale price by property type in Washington state?")

08:38:11 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:38:11 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:38:11 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:38:11 | DEBUG   | httpcore.http11 | send_request_body.complete
08:38:11 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:38:12 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:38:13 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  What is the average sale price by property type in Washington state?
──────────────────────────────────────────────────────────────
💬  Here is the average sale price by property type in Washington state:

| Property Type | Average Sale Price |
|---------------|--------------------|
| Apartment     | $261,615          |
| Condo         | $394,375          |
| House         | $883,032          |
| Townhouse     | $612,500          |

🔧  Tools: query_data  (2 steps)
     └─ Step 1: query_data



In [15]:
# Scenario 3: Schema inspection
_, _ = await ask("What datasets do you have? Show me their structure.")

08:38:14 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:38:14 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:38:14 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:38:14 | DEBUG   | httpcore.http11 | send_request_body.complete
08:38:14 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:38:15 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:38:16 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  What datasets do you have? Show me their structure.
──────────────────────────────────────────────────────────────
💬  I have two datasets available:

1. **Real Estate Listings**:
   - **ID Column**: `Listing ID`
   - **Total Rows**: 1,005
   - **Columns**:
     - Listing ID
     - Property Type
     - City
     - State
     - Bedrooms
     - Bathrooms
     - Square Footage
     - Year Built
     - List Price
     - Sale Price
     - Listing Status

2. **Marketing Campaigns**:
   - **ID Column**: `Campaign ID`
   - **Total Rows**: 998
   - **Columns**:
     - Campaign ID
     - Campaign Name
     - Channel
     - Start Date
     - End Date
     - Budget Allocated
     - Amount Spent
     - Impressions
     - Clicks
     - Conversions
     - Revenue Generated

Let me know if you'd like to explore or query any of these datasets!

🔧  Tools: inspect_schema  (2 steps)
     └─ Step 1: inspect_schema



In [16]:
# Scenario 4: Top-N query
_, _ = await ask("Show me the top 5 most expensive properties that have been sold")

08:38:17 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:38:17 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:38:17 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:38:17 | DEBUG   | httpcore.http11 | send_request_body.complete
08:38:17 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:38:19 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:38:19 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  Show me the top 5 most expensive properties that have been sold
──────────────────────────────────────────────────────────────
💬  Here are the top 5 most expensive properties that have been sold:

| Listing ID | Property Type | City          | State | Bedrooms | List Price   | Sale Price   |
|------------|---------------|---------------|-------|----------|--------------|--------------|
| LST-5504   | House         | Denver        | CO    | 5        | $1,852,000   | $1,853,000   |
| LST-5312   | House         | Fort Collins  | CO    | 5        | $1,692,000   | $1,742,000   |
| LST-5404   | House         | Boulder       | CO    | 5        | $1,651,000   | $1,649,000   |
| LST-5854   | House         | Rochester     | NY    | 4        | $1,579,000   | $1,619,000   |
| LST-5929   | House         | Sacramento    | CA    | 5        | $1,512,000   | $1,599,000   |

These properties represent the highest sale prices in the datas

In [17]:
# Scenario 5: Marketing data query
_, _ = await ask("Which marketing channel generates the most revenue on average?")

08:38:21 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:38:21 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:38:21 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:38:21 | DEBUG   | httpcore.http11 | send_request_body.complete
08:38:21 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:38:22 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:38:23 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  Which marketing channel generates the most revenue on average?
──────────────────────────────────────────────────────────────
💬  The marketing channel that generates the most revenue on average is **Email**, with an average revenue of **$142,037.07**.

🔧  Tools: query_data  (2 steps)
     └─ Step 1: query_data



In [18]:
# Scenario 6: Multi-turn conversation
session, _ = await ask("How many houses are in Texas?")
# Follow-up using the same session
import time; time.sleep(3)  # Avoid rate limits
_, _ = await ask("What's the average price there?", session=session)

08:38:24 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:38:24 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:38:24 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:38:24 | DEBUG   | httpcore.http11 | send_request_body.complete
08:38:24 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:38:24 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 429, b'Too Many Requests', [(b'Date', b'Thu, 23 Apr 2026 06:38:25 GMT'), (b'Content-Type', b'application/json; charset=utf-8'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'retry-after', b'43'), (b'x-ratelimit-timeremaining', b'43'), (b'request-context', b'appId='), (b'x-ms-error-code', b'RateLimitReached'), (b'x-ratelimit-type', b'UserByModelByMinute'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content


──────────────────────────────────────────────────────────────
❓  How many houses are in Texas?
──────────────────────────────────────────────────────────────
💬  There are **34 houses** in Texas listed in the dataset.

🔧  Tools: query_data  (2 steps)
     └─ Step 1: query_data



08:39:14 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:39:14 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:39:14 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:39:14 | DEBUG   | httpcore.http11 | send_request_body.complete
08:39:14 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:39:15 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:39:16 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  What's the average price there?
──────────────────────────────────────────────────────────────
💬  The average list price for houses in Texas is **$486,235.29**.

🔧  Tools: query_data  (2 steps)
     └─ Step 1: query_data



In [19]:
# Scenario 7: Update with preview + confirmation
import time; time.sleep(3)
session, result = await ask("Update the list price of LST-5002 to 750000")

if result.requires_confirmation:
    print("\n✅ Confirming the update...")
    time.sleep(3)
    _, confirm_result = await ask("yes", session=session)
    
    # Verify the change
    time.sleep(3)
    _, _ = await ask("What is the current list price of LST-5002?", session=session)

08:40:21 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:40:21 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:40:21 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:40:21 | DEBUG   | httpcore.http11 | send_request_body.complete
08:40:21 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:40:23 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:40:24 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  Update the list price of LST-5002 to 750000
──────────────────────────────────────────────────────────────
💬  📊 CONFIRMATION: STAGED EXCEL UPDATE (1 Row)
──────────────────────────────────────────────────
📁 File: Real Estate Listings.xlsx

 Row  | ID        | Field        | Before       → After
──────────────────────────────────────────────────
 1    | LST-5002  | List Price   | 709,000      → 750,000

Apply this change? (yes/no)

🔧  Tools: update_data  (1 step)
     └─ Step 1: update_data

⚠️   Awaiting confirmation — reply 'yes' to proceed


✅ Confirming the update...


08:40:26 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:40:26 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:40:26 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:40:26 | DEBUG   | httpcore.http11 | send_request_body.complete
08:40:26 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:40:28 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:40:28 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  yes
──────────────────────────────────────────────────────────────
💬  ✅ Successfully updated 1 row(s). (Action ID: act_99cbce97 — use this to undo if needed)

Here is the updated record:

| Listing ID | Property Type | City     | State | Bedrooms | Bathrooms | Square Footage | Year Built | List Price | Sale Price | Listing Status |
|------------|---------------|----------|-------|----------|-----------|----------------|------------|------------|------------|----------------|
| LST-5002   | House         | Bellevue | WA    | 4        | 3.5       | 2,162 sq ft    | 1960       | $750,000   | $701,000   | Sold           |

The list price has been successfully updated to **$750,000**.

🔧  Tools: query_data  (3 steps)
     └─ Step 1: execute_update
     └─ Step 2: query_data



08:40:33 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:40:33 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:40:33 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:40:33 | DEBUG   | httpcore.http11 | send_request_body.complete
08:40:33 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:40:34 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:40:35 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  What is the current list price of LST-5002?
──────────────────────────────────────────────────────────────
💬  The current list price of **LST-5002** is **$750,000**.



In [20]:
# Scenario 8: Undo the change
import time; time.sleep(3)
session, result = await ask("Undo the last change")

if result.requires_confirmation:
    print("\n✅ Confirming the undo...")
    time.sleep(3)
    _, _ = await ask("yes", session=session)

08:40:37 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:40:37 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:40:37 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:40:37 | DEBUG   | httpcore.http11 | send_request_body.complete
08:40:37 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:40:39 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:40:39 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  Undo the last change
──────────────────────────────────────────────────────────────
💬  ⏪ Undo update (action: act_99cbce97) on real_estate_listings:

Reverting values:

  Record 'LST-5002':
    List Price: 750000 → 709000 (restoring original)

Proceed with undo? (yes/no)

🔧  Tools: undo_change  (1 step)
     └─ Step 1: undo_change

⚠️   Awaiting confirmation — reply 'yes' to proceed


✅ Confirming the undo...


08:40:42 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:40:42 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:40:42 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:40:42 | DEBUG   | httpcore.http11 | send_request_body.complete
08:40:42 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:40:43 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:40:44 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  yes
──────────────────────────────────────────────────────────────
💬  ✅ Successfully undone undo_update — 1 row(s) reverted.

Here is the reverted record:

| Listing ID | Property Type | City      | State | Bedrooms | Bathrooms | Square Footage | Year Built | List Price | Sale Price | Listing Status |
|------------|---------------|-----------|-------|----------|-----------|----------------|------------|------------|------------|----------------|
| LST-5002   | House         | Bellevue  | WA    | 4        | 3.5       | 2,162 sq ft    | 1960       | $709,000   | $701,000   | Sold           |

The list price has been restored to its original value of **$709,000**.

🔧  Tools: query_data  (3 steps)
     └─ Step 1: execute_undo
     └─ Step 2: query_data



In [21]:
# Scenario 9: View change history
import time; time.sleep(3)
_, _ = await ask("Show me the recent change history")

08:40:48 | DEBUG   | httpcore.http11 | send_request_headers.started request=<Request [b'POST']>
08:40:48 | DEBUG   | httpcore.http11 | send_request_headers.complete
08:40:48 | DEBUG   | httpcore.http11 | send_request_body.started request=<Request [b'POST']>
08:40:48 | DEBUG   | httpcore.http11 | send_request_body.complete
08:40:48 | DEBUG   | httpcore.http11 | receive_response_headers.started request=<Request [b'POST']>
08:40:50 | DEBUG   | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 23 Apr 2026 06:40:50 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Vary', b'Accept-Encoding'), (b'request-context', b'appId='), (b'azureml-model-session', b'd20260402085526-2b7bd0b2b5ed'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ms-rai-invoked', b'true'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-limit-requests', 


──────────────────────────────────────────────────────────────
❓  Show me the recent change history
──────────────────────────────────────────────────────────────
💬  Here is the recent change history:

| Action ID      | Timestamp                | Operation | Dataset               | Affected Rows | Undone |
|----------------|--------------------------|-----------|-----------------------|---------------|--------|
| act_309d30ec   | 2026-04-23T03:30:59.681Z | Insert    | Real Estate Listings | 1             | Yes    |
| act_bfdaf772   | 2026-04-23T03:34:49.428Z | Insert    | Real Estate Listings | 1             | No     |
| act_c17b3a84   | 2026-04-23T03:35:38.944Z | Update    | Real Estate Listings | 1             | Yes    |
| act_3da5b2c2   | 2026-04-23T03:36:26.099Z | Update    | Real Estate Listings | 1             | No     |
| act_321e93ad   | 2026-04-23T03:52:10.579Z | Insert    | Real Estate Listings | 1             | No     |
| act_58279c6f   | 2026-04-23T03:54:52.153Z | Insert 

## 5. Examining Structured Logs

Every interaction is logged as a JSON file in the `logs/` directory.

In [22]:
from pathlib import Path
import json

logs_dir = Path('../logs')
log_files = sorted(logs_dir.glob('*.json'))

print(f"📋 Found {len(log_files)} interaction logs\n")

if log_files:
    # Show the latest log
    latest = log_files[-1]
    with open(latest) as f:
        log = json.load(f)
    
    print(f"Latest interaction:")
    print(f"  Query: {log['user_query']}")
    print(f"  Tools used: {[t['tool'] for t in log['tool_decisions']]}")
    print(f"  Response: {log['final_response'][:200]}")
    print(f"  Latency: {log['latency_ms']}ms")
    print(f"  Provider: {log['llm_provider']}")

📋 Found 187 interaction logs

Latest interaction:
  Query: Show me the recent change history
  Tools used: ['list_changes']
  Response: Here is the recent change history:

| Action ID      | Timestamp                | Operation | Dataset               | Affected Rows | Undone |
|----------------|--------------------------|-----------|
  Latency: 4406ms
  Provider: GitHubModelsProvider(model='gpt-4o')


## Summary

This notebook demonstrated:
1. **Data loading** — Excel files into pandas DataFrames
2. **Tool system** — 7 custom tools with validation and preview
3. **Agent reasoning** — ReAct loop with step-by-step reasoning
4. **Full CRUD** — Query, insert, update, delete with confirmation
5. **Undo mechanism** — Revert any past mutation
6. **Multi-turn** — Session memory for follow-up questions
7. **Structured logging** — Full traceability of every interaction